# H1 — WTE excess for GridWorld (6x6)

For each task distribution (TD): collect `(s,a,r,s')`, embed every task, and report
for **both** `(s,a,r)` and `(s,a,s')`:

- **F** — per-task noise floor (mean pairwise distance of a task's embeddings across
  seeds), averaged over the TD's tasks.
- **d(T)** — per-seed mean pairwise distance across the TD's tasks, averaged over seeds.
- **excess = d(T) − F**, for L2 and cosine.

Then the metric-property checks (non-negativity, symmetry, identity, triangle) on all
tasks pooled across TDs. Shared logic lives in `scripts/wte_h1.py`.

Goal-varying: `(s,a,r)` should rank TD1<TD2<TD3; `(s,a,s')` is a negative control (~floor) and is identical across the two rewards.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

CODE_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
for _p in (str(CODE_DIR), str(CODE_DIR / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
from environments.gridworld import GridNavi
import wte_h1 as h1

NUM_CELLS, MAX_STEPS = 6, 20
N = NUM_CELLS * NUM_CELLS * MAX_STEPS          # 720 tuples / goal
SEEDS = list(range(42, 52))                    # 10 data seeds
SPEC = dict(obs_dim=2, a_cols=1, discrete=True, n_actions=5)
FEATURE_SETS = ["sar", "sas"]
REWARDS = ["distance", "sparse"]               # grid has two reward variants
MAX_SAMPLES = None
DATA = CODE_DIR / "data" / "grid6"

def _g(opt):
    c = NUM_CELLS
    return {1: [(c-1, c-1), (c-1, c-2), (c-2, c-1), (c-2, c-2)],   # adjacent
            2: [(c//2, c//2), (c-1, c//2), (c//2, c-1), (c-1, c-1)],  # mid/edge
            3: [(0, 0), (0, c-1), (c-1, 0), (c-1, c-1)]}[opt]        # corners
TASK_DISTS = {1: _g(1), 2: _g(2), 3: _g(3)}
ALL_TASKS = [t for ts in TASK_DISTS.values() for t in ts]

def _dir(reward, seed):
    return DATA / reward / f"seed{seed}"

def collect(reward, seed):
    out = _dir(reward, seed); out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed); dr = reward == "distance"
    for tasks in TASK_DISTS.values():
        for task in tasks:
            f = out / f"goal_{task[0]}_{task[1]}.npy"
            if f.exists():
                continue
            rows = []
            for j in range(NUM_CELLS):
                for k in range(NUM_CELLS):
                    env = GridNavi(num_cells=NUM_CELLS, num_steps=MAX_STEPS,
                                   distance_reward=dr, starting_state=(j, k),
                                   possible_goals=list(tasks))
                    env.reset_task(task=task); obs, _ = env.reset()
                    for _ in range(MAX_STEPS):
                        a = int(rng.integers(env.action_space.n))
                        nobs, r, term, trunc, _ = env.step(a)
                        rows.append([obs[0], obs[1], a, r, nobs[0], nobs[1]]); obs = nobs
                        if term or trunc:
                            obs, _ = env.reset()
            np.save(f, np.asarray(rows[:N]))

def load_fn_for(reward):
    def load(task, seed):
        return np.load(_dir(reward, seed) / f"goal_{task[0]}_{task[1]}.npy")
    return load

Matplotlib is building the font cache; this may take a moment.


In [2]:
# Collect (s,a,r,s') for every task over all seeds. Safe to re-run: skips files.
for _reward in REWARDS:
    for _seed in SEEDS:
        collect(_reward, _seed)
print("collection done:", {r: len(SEEDS) for r in REWARDS}, "seeds x",
      sum(len(v) for v in TASK_DISTS.values()), "tasks")

collection done: {'distance': 10, 'sparse': 10} seeds x 12 tasks


In [3]:
# Embed, then per TD: F (mean per-task floor), d(T) (mean pairwise over seeds),
# and excess = d(T) - F, for both L2 and cosine, for (s,a,r) and (s,a,s').
EMB, _tables = {}, []
for _reward in REWARDS:
    _load = load_fn_for(_reward)
    _mu, _sd = h1.pooled_reward_stats(_load, ALL_TASKS, SEEDS,
                                      SPEC["obs_dim"], SPEC["a_cols"])
    for _fs in FEATURE_SETS:
        _emb = h1.compute_embeddings(
            TASK_DISTS, SEEDS, _load, SPEC["obs_dim"], SPEC["a_cols"], _fs, _mu, _sd,
            discrete=SPEC["discrete"], n_actions=SPEC["n_actions"],
            max_samples=MAX_SAMPLES)
        EMB[(_reward, _fs)] = _emb
        _t = h1.summarize(TASK_DISTS, SEEDS, _emb)
        _t.insert(0, "features", _fs)
        _t.insert(0, "reward", _reward)
        _tables.append(_t)
results = pd.concat(_tables, ignore_index=True)
results.round(4)

,reward,features,TD,n_tasks,d_l2,F_l2,excess_l2,d_cosine,F_cosine,excess_cosine
0,distance,sar,1,4,0.8370,0.6549,0.1821,0.1401,0.0868,0.0532
1,distance,sar,2,4,1.0715,0.6416,0.4299,0.2381,0.0855,0.1526
2,distance,sar,3,4,1.7059,0.6240,1.0819,0.5569,0.0745,0.4825
3,distance,sas,1,4,0.6795,0.6688,0.0106,0.0680,0.0658,0.0022
4,distance,sas,2,4,0.6481,0.6480,0.0001,0.0620,0.0615,0.0005
5,distance,sas,3,4,0.6490,0.6455,0.0036,0.0617,0.0613,0.0004
6,sparse,sar,1,4,0.8417,0.7819,0.0598,0.1400,0.1218,0.0182
7,sparse,sar,2,4,0.8103,0.7206,0.0897,0.1265,0.1011,0.0254
8,sparse,sar,3,4,1.3382,0.7480,0.5902,0.3415,0.1108,0.2307
9,sparse,sas,1,4,0.6795,0.6688,0.0106,0.0680,0.0658,0.0022


In [4]:
# Metric-property checks on all tasks pooled across TDs (pairwise matrix averaged
# over seeds; identity threshold delta = mean same-task-across-seeds distance).
_cls = [h1.classify(TASK_DISTS, SEEDS, _emb).assign(reward=_r, features=_f)
        for (_r, _f), _emb in EMB.items()]
classification = pd.concat(_cls, ignore_index=True)[[
    "reward", "features", "metric", "delta", "nonneg", "symmetric",
    "identity_ok", "n_collapsed", "n_pairs", "triangle_ok", "triangle_bad", "class"]]
classification.round(4)

,reward,features,metric,delta,nonneg,symmetric,identity_ok,n_collapsed,n_pairs,triangle_ok,triangle_bad,class
0,distance,sar,l2,0.6402,True,True,False,3,66,True,0/220,pseudometric
1,distance,sar,cosine,0.0822,True,True,False,3,66,False,5/220,neither
2,distance,sas,l2,0.6541,True,True,False,32,66,True,0/220,pseudometric
3,distance,sas,cosine,0.0629,True,True,False,32,66,True,0/220,pseudometric
4,sparse,sar,l2,0.7502,True,True,False,9,66,True,0/220,pseudometric
5,sparse,sar,cosine,0.1112,True,True,False,9,66,True,0/220,pseudometric
6,sparse,sas,l2,0.6541,True,True,False,32,66,True,0/220,pseudometric
7,sparse,sas,cosine,0.0629,True,True,False,32,66,True,0/220,pseudometric
